# Disease prediction machine learning project
## Overview
In this project, we aim to develop a classification model to make the work of physicians easier when predicting and or diagnosing diseases in their patients.

## Dataset

The dataset used in this project consists of 133 columns. Of which `132` represents common symptoms to diseases, and a column representing the prognosis of `42` different diseases.

## Approach
- **Load training dataset and preprocess:** Load dataset from disk, preprocess the data and split into feature matrix(X) and labels(y).
- **Model training and selection:** Train several classifieers on the training dataset and select the best.
- **Best model evaluation:** Evaluate the best model on the test set.

### Import libraries

In [ ]:
import pandas
from pathlib import Path
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

### Load training set

In [ ]:
dataset_dir = Path("/kaggle/input/disease-prediction-data").resolve()
df = pandas.read_csv(dataset_dir / "Training.csv")
df.describe()

In [ ]:
df = df.drop("Unnamed: 133", axis=1)
df["prognosis"].unique()

### Split training set into feature matrix and labels then ecocde label

In [ ]:
labels = df["prognosis"]
X = df.iloc[:, :-1]
X = X.values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
X.shape, y.shape

### Train and select best model

In [ ]:
pipeline = Pipeline([
    ("classifier", "passthrough")
])

RAND_STATE = 69

param_grid = [
    {"classifier": [AdaBoostClassifier(LogisticRegression(solver="lbfgs", multi_class="multinomial"))]},
    {"classifier": [RandomForestClassifier(random_state=RAND_STATE, n_jobs=-1)]},
    {"classifier": [GradientBoostingClassifier()],
    "classifier__learning_rate": [0.03, 0.01, 0.1, 1.0]}
]

grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1, scoring='accuracy', verbose=2)
grid.fit(X, y)
model = grid.best_estimator_
print(f"Best accuracy: {grid.best_score_}")

### Evaluate best model on test set

In [ ]:
testing_df = pandas.read_csv(dataset_dir / "Testing.csv")
y_test = label_encoder.transform(testing_df["prognosis"])
X_test = testing_df.iloc[:, :-1]
X_test = X_test.values
test_predictions = model.predict(X_test)
print(classification_report(y_test, test_predictions))